Data Generator

In [1]:
import numpy as np
import tensorflow as tf
import os
import math
import json
import time
import nltk
import pandas as pd
from sklearn.model_selection import train_test_split
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score

# Download dependensi METEOR
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)

# Load metadata
with open('../../../dataset/vocab.json', 'r') as f:
    vocab_data = json.load(f)

VOCAB_SIZE      = vocab_data['vocab_size']
MAX_LENGTH      = vocab_data['max_length']
WORD_TO_INDEX   = vocab_data['word_to_index']
INDEX_TO_WORD   = {int(k): v for k, v in vocab_data['index_to_word'].items()}
FEATURE_DIR     = "../../../dataset/features"

In [2]:
class CaptionDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, df, batch_size, feature_dir, word_to_index, max_length):
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.feature_dir = feature_dir
        self.word_to_index = word_to_index
        self.max_length = max_length

    def __len__(self):
        return math.ceil(len(self.df) / self.batch_size)

    def __getitem__(self, idx):
        batch_df = self.df.iloc[idx * self.batch_size : (idx + 1) * self.batch_size]
        
        features_input = []
        captions_input = []
        targets = []
        
        for _, row in batch_df.iterrows():
            feat_path = os.path.join(self.feature_dir, row['image'] + ".npy")
            feature = np.load(feat_path)
            
            seq = [self.word_to_index.get(w, 0) for w in str(row['caption']).split()]
            
            in_seq = seq[:-1]   
            out_seq = seq[1:]   
            
            features_input.append(feature)
            captions_input.append(in_seq)
            targets.append(out_seq)
            
        captions_input = tf.keras.preprocessing.sequence.pad_sequences(
            captions_input, maxlen=self.max_length, padding='post'
        )
        targets = tf.keras.preprocessing.sequence.pad_sequences(
            targets, maxlen=self.max_length, padding='post'
        )
        
        inputs = {
            "image_input": np.array(features_input),
            "caption_input": np.array(captions_input)
        }
        
        return inputs, np.array(targets)

Build LSTM Model

In [3]:
def build_lstm_model(num_layers, hidden_size, embed_dim=256):
    # Input 1: CNN Feature
    image_input = tf.keras.Input(shape=(2048,), name="image_input")
    image_emb = tf.keras.layers.Dense(embed_dim, activation='relu', name="proj_dense")(image_input)
    image_emb = tf.keras.layers.Reshape((1, embed_dim))(image_emb)

    # Input 2: Caption Sequence
    caption_input = tf.keras.Input(shape=(MAX_LENGTH,), name="caption_input")
    caption_emb = tf.keras.layers.Embedding(VOCAB_SIZE, embed_dim, mask_zero=True, name="embedding")(caption_input)

    # Pre-inject
    merged = tf.keras.layers.Concatenate(axis=1)([image_emb, caption_emb])

    # Recurrent Layers
    x = merged
    for i in range(num_layers):
        x = tf.keras.layers.LSTM(hidden_size, return_sequences=True, name=f"lstm_{i}")(x)

    # Output Layer -> timestep teks = index 1 dst.
    x = tf.keras.layers.Lambda(lambda t: t[:, 1:, :])(x)
    
    output = tf.keras.layers.TimeDistributed(
        tf.keras.layers.Dense(VOCAB_SIZE, activation='softmax'),
        name="time_dist_out"
    )(x)

    model = tf.keras.Model(inputs=[image_input, caption_input], outputs=output)
    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

Evaluation Function

In [4]:
def generate_caption_keras(model, image_feature, max_length, word_to_index, index_to_word):
    in_text = '<start>'
    for i in range(max_length):
        sequence = [word_to_index.get(w, 0) for w in in_text.split()]
        sequence = tf.keras.preprocessing.sequence.pad_sequences([sequence], maxlen=max_length, padding='post')
        
        inputs = {
            "image_input": np.expand_dims(image_feature, axis=0),
            "caption_input": sequence
        }
        yhat = model.predict(inputs, verbose=0)
        
        current_len = len(in_text.split())
        yhat_idx = np.argmax(yhat[0, current_len - 1, :])
        word = index_to_word.get(yhat_idx, '')
        
        if word == '<end>' or not word:
            break
        in_text += ' ' + word
        
    return in_text.replace('<start> ', '').strip()

def evaluate_metrics(model, test_df, feature_dir, word_to_index, index_to_word, max_length):
    actual, predicted = [], []
    test_grouped = test_df.groupby('image')['caption'].apply(list).to_dict()
    
    print("Mengevaluasi set test...")
    for img_name, captions in list(test_grouped.items()): 
        feat_path = os.path.join(feature_dir, img_name + ".npy")
        feature = np.load(feat_path)
        
        yhat = generate_caption_keras(model, feature, max_length, word_to_index, index_to_word)
        references = [c.replace('<start> ', '').replace(' <end>', '').split() for c in captions]
        
        actual.append(references)
        predicted.append(yhat.split())
    
    smoothie = SmoothingFunction().method4
    bleu_4 = corpus_bleu(actual, predicted, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothie)
    
    meteor_scores = [meteor_score(ref, hyp) for ref, hyp in zip(actual, predicted)]
    return bleu_4, np.mean(meteor_scores)

Data Preparation and Splitting

In [5]:
# Data Preparation
df_captions = pd.read_csv("../../../dataset/captions.txt")
df_captions = df_captions.dropna()
df_captions['caption'] = "<start> " + df_captions['caption'].str.lower() + " <end>"

# Splitting
unique_images = df_captions['image'].unique()

train_val_imgs, test_imgs = train_test_split(
    unique_images, 
    test_size=1000, 
    random_state=42
)

train_imgs, val_imgs = train_test_split(
    train_val_imgs, 
    test_size=1000, 
    random_state=42
)

# Filter dataframe utama
train_df = df_captions[df_captions['image'].isin(train_imgs)]
val_df = df_captions[df_captions['image'].isin(val_imgs)]
test_df = df_captions[df_captions['image'].isin(test_imgs)]

print(f"Distribusi Gambar:")
print(f"- Train : {len(train_imgs)} gambar -> {len(train_df)} baris caption")
print(f"- Val   : {len(val_imgs)} gambar -> {len(val_df)} baris caption")
print(f"- Test  : {len(test_imgs)} gambar -> {len(test_df)} baris caption")

Distribusi Gambar:
- Train : 6091 gambar -> 30455 baris caption
- Val   : 1000 gambar -> 5000 baris caption
- Test  : 1000 gambar -> 5000 baris caption


Training Loop

In [6]:
# Hyperparameters
layers_variations = [1, 2, 3]
hidden_variations = [256, 512]

# Instansiasi Generator
train_gen = CaptionDataGenerator(train_df, 128, FEATURE_DIR, WORD_TO_INDEX, MAX_LENGTH)
val_gen = CaptionDataGenerator(val_df, 128, FEATURE_DIR, WORD_TO_INDEX, MAX_LENGTH)

for n_layers in layers_variations:
    for h_size in hidden_variations:
        tf.keras.backend.clear_session()
        
        version_name = f"lstm_l{n_layers}_h{h_size}"
        save_path = f"../weights/"
        os.makedirs(save_path, exist_ok=True)
        
        print(f"\n[{version_name}] Memulai Pelatihan...")
        model = build_lstm_model(num_layers=n_layers, hidden_size=h_size)
        
        start_time = time.time()
        
        # Training
        history = model.fit(
            train_gen,
            validation_data=val_gen,
            epochs=5
        )
        
        exec_time = time.time() - start_time
        
        # Evaluasi Metrik
        bleu4, meteor = evaluate_metrics(model, test_df, FEATURE_DIR, WORD_TO_INDEX, INDEX_TO_WORD, MAX_LENGTH)
        
        print(f"[{version_name}] Waktu: {exec_time:.2f}s | BLEU-4: {bleu4:.4f} | METEOR: {meteor:.4f}")
        
        # Simpan bobot
        weights_file = os.path.join(save_path, f"{version_name}.weights.h5")
        model.save_weights(weights_file)
        
        # Simpan metadata
        metadata = {
            "num_layers": n_layers,
            "hidden_size": h_size,
            "execution_time_seconds": exec_time,
            "bleu_4": bleu4,
            "meteor": meteor,
            "history": history.history
        }
        
        metadata_file = os.path.join(save_path, f"{version_name}_metadata.json")
        with open(metadata_file, "w") as f:
            json.dump(metadata, f, indent=4)
            
        print(f"[{version_name}] Data disimpan di {save_path}")



[lstm_l1_h256] Memulai Pelatihan...


d:\GitHub\CNN-RNN-LSTM-Scratch\env\Lib\site-packages\keras\src\layers\layer.py:1035: UserWarning: Layer 'lambda' (of type Lambda) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
d:\GitHub\CNN-RNN-LSTM-Scratch\env\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5


d:\GitHub\CNN-RNN-LSTM-Scratch\env\Lib\site-packages\keras\src\layers\layer.py:1035: UserWarning: Layer 'lambda' (of type Lambda) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


238/238 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.8611 - loss: 2.4660

...

238/238 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.8610 - loss: 1.7275

d:\GitHub\CNN-RNN-LSTM-Scratch\env\Lib\site-packages\keras\src\layers\layer.py:1035: UserWarning: Layer 'lambda' (of type Lambda) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


238/238 ━━━━━━━━━━━━━━━━━━━━ 1146s 5s/step - accuracy: 0.8814 - loss: 1.1023 - val_accuracy: 0.8880 - val_loss: 0.8780
Epoch 2/5
238/238 ━━━━━━━━━━━━━━━━━━━━ 1055s 4s/step - accuracy: 0.8886 - loss: 0.8479 - val_accuracy: 0.8898 - val_loss: 0.8314
Epoch 3/5
238/238 ━━━━━━━━━━━━━━━━━━━━ 1021s 4s/step - accuracy: 0.8911 - loss: 0.7875 - val_accuracy: 0.8926 - val_loss: 0.7451
Epoch 4/5
238/238 ━━━━━━━━━━━━━━━━━━━━ 1070s 4s/step - accuracy: 0.8942 - loss: 0.7190 - val_accuracy: 0.8939 - val_loss: 0.7244
Epoch 5/5
238/238 ━━━━━━━━━━━━━━━━━━━━ 1001s 4s/step - accuracy: 0.8954 - loss: 0.6940 - val_accuracy: 0.8949 - val_loss: 0.7030
Mengevaluasi set test...


d:\GitHub\CNN-RNN-LSTM-Scratch\env\Lib\site-packages\keras\src\layers\layer.py:1035: UserWarning: Layer 'lambda' (of type Lambda) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


[lstm_l3_h512] Waktu: 5292.38s | BLEU-4: 0.0594 | METEOR: 0.1623
[lstm_l3_h512] Data disimpan di ../weights/
